# Tokenizer and span alignment — LAB 10

**AI-assisted theo ủy quyền rõ ràng của người dùng.** AI triển khai alignment, tính toán và kiểm thử; không nhận là bài do sinh viên tự làm và không khẳng định giảng viên đã chấp thuận ngoại lệ mục 29. Các ví dụ nhân tạo dưới đây kiểm tra logic, không phải kết quả chất lượng model.

Chạy tuần tự từ kernel sạch tại `lab10`. Input: fast tokenizer cùng revision của lượt full run và các ví dụ đã ghi rõ. Thao tác: inspect arrays, map offsets và assert alignment/window. Output: `.runtime/full_run/tokenizer_alignment.json`. Mặc định đọc model snapshot từ `.runtime/full_run/bootstrap.json`; biến `LAB10_MODEL` chỉ dùng khi chủ động kiểm tra tokenizer khác, và report ghi đúng tokenizer thực dùng.

## 10. Tokenization của QA

`input_ids` là ID từ vựng của từng token trong pair, không phải character index. `attention_mask=1` cho token thật và `0` cho padding. `sequence_ids=0` chỉ question, `1` chỉ context, `None` chỉ special/padding; chỉ context được phép là đáp án. `token_type_ids` là segment IDs khi tokenizer/model hỗ trợ: BERT thường có, DistilBERT không dùng; code chỉ hiển thị nếu có, không tự tạo rồi truyền vào mọi model.

`overflow_to_sample_mapping` nối mỗi window với sample gốc; một sample có thể tạo nhiều feature. Cell tiếp theo in đầy đủ arrays và token strings để kiểm tra trực tiếp.

In [1]:
# Input: cwd lab10, optional LAB10_MODEL local path or Hugging Face tokenizer ID.
# Operation: constrain runtime/cache to lab10, load a fast tokenizer; output: cache, no result report yet.
from pathlib import Path
import os, sys, json
ROOT = Path.cwd().resolve()
assert (ROOT / "lab_10_extractive_question_answering.md").is_file(), "Run kernel from lab10."
RUNTIME = (ROOT / ".runtime" / "full_run").resolve()
RUNTIME.mkdir(parents=True, exist_ok=True)
os.environ["LAB10_RUNTIME"] = str(RUNTIME)
for key, folder in {"HF_HOME":"hf_cache", "HF_DATASETS_CACHE":"hf_cache/datasets", "MPLCONFIGDIR":"matplotlib"}.items():
    os.environ[key] = str(RUNTIME / folder)
os.environ["HF_HUB_CACHE"] = str(RUNTIME / "hf_cache" / "hub")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
sys.path.insert(0, str(ROOT))
from qa_model import Config, align_span
from transformers import AutoTokenizer
bootstrap_path = RUNTIME / "bootstrap.json"
bootstrap = json.loads(bootstrap_path.read_text(encoding="utf-8")) if bootstrap_path.exists() else {}
TOKENIZER_NAME = os.environ.get("LAB10_MODEL") or bootstrap.get("model_source", "distilbert-base-uncased")
TOKENIZER_REVISION = bootstrap.get("model_revision") if "LAB10_MODEL" not in os.environ else None
config = Config()
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME, use_fast=True,
                                          cache_dir=str(RUNTIME / "hf_cache" / "hub"),
                                          local_files_only=Path(TOKENIZER_NAME).is_dir(),
                                          revision=TOKENIZER_REVISION)
assert tokenizer.is_fast, "A fast tokenizer is required for offsets."
print({"tokenizer": TOKENIZER_NAME, "runtime": str(RUNTIME), "model_input_names": tokenizer.model_input_names})

{'tokenizer': 'C:\\Users\\tasdus_117\\Desktop\\NLP\\lab10\\.runtime\\full_run\\hf_cache\\hub\\models--distilbert-base-uncased\\snapshots\\12040accade4e8a0f71eabdb258fecc2e7e948be', 'runtime': 'C:\\Users\\tasdus_117\\Desktop\\NLP\\lab10\\.runtime\\full_run', 'model_input_names': ['input_ids', 'token_type_ids', 'attention_mask']}


In [2]:
# Input: question/context strings below. Operation: pair-tokenize and display arrays. Output: in-memory arrays, printed tokens.
question = "What is the capital of France?"
context = "The capital of France is Paris."
encoded = tokenizer(question, context, return_offsets_mapping=True,
                    return_overflowing_tokens=True, truncation="only_second", max_length=384, stride=128)
arrays = {key: encoded[key] for key in ("input_ids", "attention_mask", "offset_mapping", "overflow_to_sample_mapping")}
if "token_type_ids" in encoded:
    arrays["token_type_ids"] = encoded["token_type_ids"]
else:
    print("This tokenizer does not use token_type_ids.")
arrays["sequence_ids"] = [encoded.sequence_ids(i) for i in range(len(encoded["input_ids"]))]
print(json.dumps(arrays, indent=2))
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"][0]))

{
  "input_ids": [
    [
      101,
      2054,
      2003,
      1996,
      3007,
      1997,
      2605,
      1029,
      102,
      1996,
      3007,
      1997,
      2605,
      2003,
      3000,
      1012,
      102
    ]
  ],
  "attention_mask": [
    [
      1,
      1,
      1,
      1,
      1,
      1,
      1,
      1,
      1,
      1,
      1,
      1,
      1,
      1,
      1,
      1,
      1
    ]
  ],
  "offset_mapping": [
    [
      [
        0,
        0
      ],
      [
        0,
        4
      ],
      [
        5,
        7
      ],
      [
        8,
        11
      ],
      [
        12,
        19
      ],
      [
        20,
        22
      ],
      [
        23,
        29
      ],
      [
        29,
        30
      ],
      [
        0,
        0
      ],
      [
        0,
        3
      ],
      [
        4,
        11
      ],
      [
        12,
        14
      ],
      [
        15,
        21
      ],
      [
        22,
        24
      

## 11. Offset Mapping
Offsets are half-open character intervals `[start, end)` into the original context for context tokens. A word can contain several subword tokens; character, word, and pair-token indices therefore differ.

## 12. Calculation Exercise 3: Character Span

In [3]:
# Input: Paris annotation and actual tokenizer offsets. Operation: calculate/reference-align span. Output: in-memory calculation and printed evidence.
answer_text = "Paris"
answer_start = context.index(answer_text)
answer_end = answer_start + len(answer_text)
cls_index = encoded["input_ids"][0].index(tokenizer.cls_token_id)
start_token, end_token = align_span(encoded["offset_mapping"][0], encoded.sequence_ids(0), answer_start, answer_text, cls_index)
calculation = {"first_character":answer_start, "last_character_inclusive":answer_end-1,
               "character_span_half_open":[answer_start,answer_end], "word_index":context.split().index("Paris."),
               "token_start":start_token, "token_end_inclusive":end_token,
               "tokens":tokenizer.convert_ids_to_tokens(encoded["input_ids"][0][start_token:end_token+1])}
assert context[answer_start:answer_end] == answer_text
print(calculation)

{'first_character': 25, 'last_character_inclusive': 29, 'character_span_half_open': [25, 30], 'word_index': 5, 'token_start': 14, 'token_end_inclusive': 14, 'tokens': ['paris']}


## 13. Truncation
The answer below is deliberately near the end of a long input. A single truncated window loses it; overlapping windows recover it. An out-of-window training target maps to CLS, which is a feature-level sentinel rather than evidence of an unanswerable SQuAD 1 question.

In [4]:
# Input: long synthetic context with answer near end. Operation: compare truncation versus overflow. Output: window_labels with true sample mapping.
long_context = ("Unrelated background information fills this sentence. " * 80) + context
long_start = long_context.rindex("Paris")
short = tokenizer(question, long_context, max_length=128, truncation="only_second", return_offsets_mapping=True)
short_cls = short["input_ids"].index(tokenizer.cls_token_id)
assert align_span(short["offset_mapping"], short.sequence_ids(), long_start, "Paris", short_cls) == (short_cls, short_cls)
windows = tokenizer(question, long_context, max_length=128, stride=64, truncation="only_second",
                    return_offsets_mapping=True, return_overflowing_tokens=True)
window_labels = []
for i, offsets in enumerate(windows["offset_mapping"]):
    cls = windows["input_ids"][i].index(tokenizer.cls_token_id)
    span = align_span(offsets, windows.sequence_ids(i), long_start, "Paris", cls)
    window_labels.append({"window":i,"sample":windows["overflow_to_sample_mapping"][i],"span":span,"contains_answer":span != (cls,cls)})
assert any(row["contains_answer"] for row in window_labels)
assert any(not row["contains_answer"] for row in window_labels)
print(json.dumps(window_labels, indent=2))

[
  {
    "window": 0,
    "sample": 0,
    "span": [
      0,
      0
    ],
    "contains_answer": false
  },
  {
    "window": 1,
    "sample": 0,
    "span": [
      0,
      0
    ],
    "contains_answer": false
  },
  {
    "window": 2,
    "sample": 0,
    "span": [
      0,
      0
    ],
    "contains_answer": false
  },
  {
    "window": 3,
    "sample": 0,
    "span": [
      0,
      0
    ],
    "contains_answer": false
  },
  {
    "window": 4,
    "sample": 0,
    "span": [
      0,
      0
    ],
    "contains_answer": false
  },
  {
    "window": 5,
    "sample": 0,
    "span": [
      0,
      0
    ],
    "contains_answer": false
  },
  {
    "window": 6,
    "sample": 0,
    "span": [
      0,
      0
    ],
    "contains_answer": false
  },
  {
    "window": 7,
    "sample": 0,
    "span": [
      0,
      0
    ],
    "contains_answer": false
  },
  {
    "window": 8,
    "sample": 0,
    "span": [
      0,
      0
    ],
    "contains_answer": false
  },
  {
    

## 15. Core Implementation
Section 10 prints every requested array. Here the complete overflowing batch is inspected, including original sample mapping. Use stride 128 with max length 384 as in the handout; the controlled length experiment uses common stride 64 so its 128-token run remains valid.

In [5]:
# Input: same long context. Operation: inspect handout max_length384/stride128 arrays. Output: printed arrays only.
core = tokenizer(question, long_context, truncation="only_second", max_length=384, stride=128,
                 return_overflowing_tokens=True, return_offsets_mapping=True)
for key in ("input_ids", "attention_mask", "offset_mapping", "overflow_to_sample_mapping"):
    print(key, core[key])
print("sequence_ids", [core.sequence_ids(i) for i in range(len(core["input_ids"]))])
if "token_type_ids" in core: print("token_type_ids", core["token_type_ids"])

input_ids [[101, 2054, 2003, 1996, 3007, 1997, 2605, 1029, 102, 15142, 4281, 2592, 17469, 2023, 6251, 1012, 15142, 4281, 2592, 17469, 2023, 6251, 1012, 15142, 4281, 2592, 17469, 2023, 6251, 1012, 15142, 4281, 2592, 17469, 2023, 6251, 1012, 15142, 4281, 2592, 17469, 2023, 6251, 1012, 15142, 4281, 2592, 17469, 2023, 6251, 1012, 15142, 4281, 2592, 17469, 2023, 6251, 1012, 15142, 4281, 2592, 17469, 2023, 6251, 1012, 15142, 4281, 2592, 17469, 2023, 6251, 1012, 15142, 4281, 2592, 17469, 2023, 6251, 1012, 15142, 4281, 2592, 17469, 2023, 6251, 1012, 15142, 4281, 2592, 17469, 2023, 6251, 1012, 15142, 4281, 2592, 17469, 2023, 6251, 1012, 15142, 4281, 2592, 17469, 2023, 6251, 1012, 15142, 4281, 2592, 17469, 2023, 6251, 1012, 15142, 4281, 2592, 17469, 2023, 6251, 1012, 15142, 4281, 2592, 17469, 2023, 6251, 1012, 15142, 4281, 2592, 17469, 2023, 6251, 1012, 15142, 4281, 2592, 17469, 2023, 6251, 1012, 15142, 4281, 2592, 17469, 2023, 6251, 1012, 15142, 4281, 2592, 17469, 2023, 6251, 1012, 15142, 4281,

## 16. Answer Span Alignment
The executable checks exercise answers beginning and ending inside a real tokenizer token. Such a token encloses the annotated substring; decoding the whole token can include extra characters. This is why offset slicing and exact character boundaries must be distinguished.

In [6]:
# Input: real subword offsets and artificial substrings inside tokens. Operation: assert enclosing-token alignment. Output: edge_checks in memory.
pair = tokenizer("Which word?", "Extraordinary biodiversity flourishes.", return_offsets_mapping=True)
choices = [(i,a,b) for i,(a,b) in enumerate(pair["offset_mapping"]) if pair.sequence_ids()[i] == 1 and b-a >= 3]
assert choices, "Need a context token at least three characters wide for this check."
i,a,b = choices[0]
edge_context = "Extraordinary biodiversity flourishes."
edge_checks = []
for label,lo,hi in [("starts_inside_token",a+1,b),("ends_inside_token",a,b-1),("both_inside_token",a+1,b-1)]:
    text = edge_context[lo:hi]
    span = align_span(pair["offset_mapping"],pair.sequence_ids(),lo,text,pair["input_ids"].index(tokenizer.cls_token_id))
    assert span == (i,i)
    edge_checks.append({"case":label,"answer":text,"character_span":[lo,hi],"token_span":span})


### Kiểm tra bổ sung cho mục 16: partial span và nhiều sample

Một đáp án có start trong window nhưng end ngoài window phải dùng CLS; không gán một nửa span. Kiểm tra đối xứng ở mép trái. Overflow tạo nhiều feature/sample; phải dùng `overflow_to_sample_mapping`, không zip feature với example theo index.

In [7]:
# Input: previous long context/windows plus a second short example.
# Operation: assert partial-span rejection and overflow sample association. Output: extra_checks for the final JSON report.
short_context_tokens = [(i,a,b) for i,(a,b) in enumerate(short["offset_mapping"]) if short.sequence_ids()[i] == 1]
last_i, last_a, last_b = short_context_tokens[-1]
partial_right = long_context[last_a:last_b+12]
assert len(partial_right) > last_b-last_a
assert align_span(short["offset_mapping"], short.sequence_ids(), last_a, partial_right, short_cls) == (short_cls, short_cls)
later = next(i for i in range(len(windows["input_ids"]))
             if min(a for (a,b),sid in zip(windows["offset_mapping"][i],windows.sequence_ids(i)) if sid == 1) > 0)
first_a, first_b = next((a,b) for (a,b),sid in zip(windows["offset_mapping"][later],windows.sequence_ids(later)) if sid == 1)
partial_left_start = first_a-1
partial_left = long_context[partial_left_start:first_b]
later_cls = windows["input_ids"][later].index(tokenizer.cls_token_id)
assert align_span(windows["offset_mapping"][later],windows.sequence_ids(later),partial_left_start,partial_left,later_cls) == (later_cls,later_cls)
batch_contexts = [long_context, "Rome is in Italy."]
batch_questions = [question, "Where is Rome?"]
batch_answers = [(long_start,"Paris"),(11,"Italy")]
batch = tokenizer(batch_questions,batch_contexts,max_length=128,stride=64,truncation="only_second",
                  return_offsets_mapping=True,return_overflowing_tokens=True,padding="max_length")
assert set(batch["overflow_to_sample_mapping"]) == {0,1}
assert batch["overflow_to_sample_mapping"].count(0) > 1
batch_checks = []
for i, sample in enumerate(batch["overflow_to_sample_mapping"]):
    char_start, text = batch_answers[sample]
    assert batch_contexts[sample][char_start:char_start+len(text)] == text
    cls = batch["input_ids"][i].index(tokenizer.cls_token_id)
    s,e = align_span(batch["offset_mapping"][i],batch.sequence_ids(i),char_start,text,cls)
    if (s,e) != (cls,cls):
        assert batch.sequence_ids(i)[s] == batch.sequence_ids(i)[e] == 1
        assert batch["attention_mask"][i][s] == batch["attention_mask"][i][e] == 1
        assert batch_contexts[sample][batch["offset_mapping"][i][s][0]:batch["offset_mapping"][i][e][1]] == text
    batch_checks.append({"feature":i,"sample":sample,"span":[s,e],"contains_answer":(s,e)!=(cls,cls)})
assert any(x["sample"] == 0 and x["contains_answer"] for x in batch_checks)
assert any(x["sample"] == 1 and x["contains_answer"] for x in batch_checks)
extra_checks = {"partial_right_rejected":True,"partial_left_rejected":True,"batch_overflow":batch_checks}
print(json.dumps(extra_checks,indent=2))

{
  "partial_right_rejected": true,
  "partial_left_rejected": true,
  "batch_overflow": [
    {
      "feature": 0,
      "sample": 0,
      "span": [
        0,
        0
      ],
      "contains_answer": false
    },
    {
      "feature": 1,
      "sample": 0,
      "span": [
        0,
        0
      ],
      "contains_answer": false
    },
    {
      "feature": 2,
      "sample": 0,
      "span": [
        0,
        0
      ],
      "contains_answer": false
    },
    {
      "feature": 3,
      "sample": 0,
      "span": [
        0,
        0
      ],
      "contains_answer": false
    },
    {
      "feature": 4,
      "sample": 0,
      "span": [
        0,
        0
      ],
      "contains_answer": false
    },
    {
      "feature": 5,
      "sample": 0,
      "span": [
        0,
        0
      ],
      "contains_answer": false
    },
    {
      "feature": 6,
      "sample": 0,
      "span": [
        0,
        0
      ],
      "contains_answer": false
    },
    {


In [8]:
# Input: actual arrays and passing checks above. Operation: serialize report. Output: .runtime/full_run/tokenizer_alignment.json (not training results).
import importlib.metadata
report = {"tokenizer":TOKENIZER_NAME,"tokenizer_revision":TOKENIZER_REVISION,"arrays":arrays,"calculation":calculation,
          "long_input_windows":window_labels,"edge_checks":edge_checks,"extra_checks":extra_checks,
          "versions":{name:importlib.metadata.version(name) for name in ["transformers","tokenizers"]}}
report_path = RUNTIME / "tokenizer_alignment.json"
report_path.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding="utf-8")
print(str(report_path))

C:\Users\tasdus_117\Desktop\NLP\lab10\.runtime\full_run\tokenizer_alignment.json


## 29. AI Usage

- **AI-assisted code:** notebook này và `align_span` trong [qa_model.py](qa_model.py).
- **What was generated:** inspect arrays, phép tính character/token, alignment và synthetic edge checks, code ghi report và giải thích.
- **What was modified:** tận dụng notebook trước, giữ đúng số/tên mục, dùng runtime full_run và tokenizer revision từ bootstrap của cùng lượt huấn luyện.
- **What was verified:** các assert thực thi và arrays thật có trong output notebook/report sau khi chạy; không dùng synthetic tests làm bằng chứng chất lượng QA. Không gọi việc chạy code AI là sinh viên tự triển khai.

Người dùng đã cho phép AI làm toàn bộ mục29 trong yêu cầu mới. Khai báo này không thay đổi đề và không khẳng định giảng viên chấp thuận ngoại lệ đó.

## 30. Reproducibility

JSON report lưu tokenizer thực dùng, revision, versions, arrays và các edge checks. Mặc định model snapshot/revision trùng bootstrap của full run; nếu đặt LAB10_MODEL khác thì report phản ánh việc đó, không gán kết quả tokenizer khác cho run DistilBERT. Xem [qa_finetuning.ipynb](qa_finetuning.ipynb), mục30, để đối chiếu data splits, hardware, checkpoint và thời gian đo.

API references: [Hugging Face tokenizer](https://huggingface.co/docs/transformers/main_classes/tokenizer), [QA task guide](https://huggingface.co/docs/transformers/tasks/question_answering).